# 02 — Limpieza y Saneamiento (Entrega 2)

Objetivo: transformar `movements_raw.csv` + `catalog_raw.csv` en el dataset analítico
`inventory_v1.csv` (25,819 filas × 17 columnas), narrando y justificando cada regla.

Este notebook es **autocontenido**: carga sus propios datos (no depende de que
`01_perfilado.ipynb` haya dejado variables en el kernel) y reimplementa, en pandas,
la misma lógica que `src/preprocessing.py` (la fuente de verdad reproducible del
pipeline, documentada en `docs/data_dictionary.md` y `docs/bitacora_entregas.md`).
La última sección compara el resultado de este notebook contra el CSV oficial
para verificar que ambas implementaciones concuerdan.

Reglas de limpieza aplicadas (P1–P6, ver `docs/bitacora_entregas.md`):
1. **P4** — Tipos de dato explícitos (evita inferencia inconsistente)
2. **P1** — Outlier calórico (umbral físico 900 kcal/100g)
3. **P2** — Homologación de categorías + derivación de ubicación física
4. **P5** — Nutriscore faltante ('Falta Dato' → nulo estructural)
5. **P6** — Deduplicación de eventos (por `event_id`)
6. **P3** — Integridad referencial (eventos huérfanos sin producto en catálogo)
7. JOIN `movements` × `catalog`
8. Derivadas: `action_type`, `dias_para_vencer`
9. Exportación + bitácora de transformaciones
10. Verificación contra el pipeline oficial (`src/preprocessing.py`)

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

os.makedirs('../data/interim', exist_ok=True)

# Carga independiente — este notebook no asume que 01_perfilado.ipynb corrió antes
catalog = pd.read_csv('../data/raw/catalog_raw.csv')
movements = pd.read_csv('../data/raw/movements_raw.csv')

log = {
    'timestamp_ejecucion': datetime.now().isoformat(),
    'registros_iniciales': {'catalog': len(catalog), 'movements': len(movements)},
    'transformaciones': {},
}

print(f'Catálogo:    {catalog.shape[0]} filas × {catalog.shape[1]} columnas')
print(f'Movimientos: {movements.shape[0]} filas × {movements.shape[1]} columnas')
print(f'\nColumnas catálogo: {list(catalog.columns)}')
print(f'Columnas movimientos: {list(movements.columns)}')

## Paso 1 (P4) — Tipos de dato explícitos

Se castea explícitamente antes de cualquier otra operación: evita que pandas infiera
tipos inconsistentes entre columnas (ej. `product_id` como float por un nulo
accidental) y deja `timestamp`/`expiry_date` listos para aritmética de fechas.
`nutriscore == 'Falta Dato'` se convierte a nulo aquí mismo — es un *sentinel*
literal, no un valor válido de la escala A–E.

In [ ]:
movements['product_id'] = movements['product_id'].astype('Int64')
movements['household_id'] = movements['household_id'].astype('Int64')
movements['quantity'] = movements['quantity'].astype('Int64')
movements['timestamp'] = pd.to_datetime(movements['timestamp'], errors='coerce')
movements['expiry_date'] = pd.to_datetime(movements['expiry_date'], errors='coerce')
movements['event_type'] = movements['event_type'].astype('category')
movements['classification'] = movements['classification'].astype('category')

catalog['product_id'] = catalog['product_id'].astype('Int64')
catalog['calories_100g'] = pd.to_numeric(catalog['calories_100g'], errors='coerce')
catalog['proteins_100g'] = pd.to_numeric(catalog['proteins_100g'], errors='coerce')
catalog['carbs_100g'] = pd.to_numeric(catalog['carbs_100g'], errors='coerce')
catalog['category'] = catalog['category'].astype('Int64')
catalog['nutriscore'] = catalog['nutriscore'].replace('Falta Dato', np.nan)

print('Tipos en movements:')
print(movements.dtypes)
print('\nTipos en catalog:')
print(catalog.dtypes)

## Paso 2 (P1) — Outlier calórico

Umbral físico: 900 kcal/100g (el máximo teórico plausible para un alimento sólido
es ≈900, correspondiente a grasa pura). Cualquier valor por encima es un error de
captura, no una variación real. Se filtra a nivel de **catálogo** (no evento por
evento) porque `calories_100g` es un atributo del producto, no del movimiento.

In [ ]:
CALORIE_THRESHOLD = 900

outliers_cal = catalog[catalog['calories_100g'] > CALORIE_THRESHOLD]
n_outliers = len(outliers_cal)
print(f'Outliers detectados en calories_100g (>{CALORIE_THRESHOLD}): {n_outliers}')
if n_outliers > 0:
    print(outliers_cal[['product_id', 'product_name', 'calories_100g']])

n_cat_antes = len(catalog)
catalog = catalog[catalog['calories_100g'].isna() | (catalog['calories_100g'] <= CALORIE_THRESHOLD)]
n_cat_post_p1 = len(catalog)

print(f'\nCatálogo: {n_cat_antes} -> {n_cat_post_p1}')

log['transformaciones']['P1_outlier_calorico'] = {
    'umbral_calories_100g': CALORIE_THRESHOLD,
    'outliers_detectados': n_outliers,
    'outliers_removidos': n_outliers,
    'registros_catalog_despues': n_cat_post_p1,
}

## Paso 3 (P2) — Homologación de categorías + ubicación física

`category` en el catálogo crudo es el `department_id` numérico de Instacart (4, 16,
7, 3, 1, 20) — no es legible ni tiene significado de negocio por sí mismo. Se mapea
a un nombre de categoría en castellano (`category_name`) **y**, a partir del mismo
`department_id`, se deriva `location` (Refrigerador / Despensa / Estante): la
ubicación física no se observó directamente en la simulación, se infiere de qué
tipo de alimento es (perecible → Refrigerador, bebidas → Estante, etc.). Este es el
mismo mapeo que usa `src/preprocessing.py` — mantenerlo idéntico es lo que permite
comparar ambas implementaciones al final del notebook.

In [ ]:
CATEGORY_MAP = {
    4: 'Frutas y Verduras', 16: 'Lacteos y Refrigerados', 7: 'Bebidas',
    3: 'Panaderia y Granos', 1: 'Despensa General', 20: 'Congelados',
}
LOCATION_MAP = {
    4: 'Refrigerador', 16: 'Refrigerador', 7: 'Estante',
    3: 'Despensa', 1: 'Despensa', 20: 'Refrigerador',
}

catalog['category_name'] = catalog['category'].map(CATEGORY_MAP)
catalog['category_name'] = catalog['category_name'].fillna('Categoria_' + catalog['category'].astype(str))
catalog['location'] = catalog['category'].map(LOCATION_MAP).fillna('Estante')

n_cats_distintas = catalog['category_name'].nunique()
print(f"dept_ids distintos: {catalog['category'].nunique()} -> category_name: {n_cats_distintas}")
print(catalog['category_name'].value_counts())
print('\nUbicación derivada:')
print(catalog['location'].value_counts())

log['transformaciones']['P2_homologacion_categorias'] = {
    'dept_ids_distintos': int(catalog['category'].nunique()),
    'category_names_finales': int(n_cats_distintas),
    'mapeo_category_name': CATEGORY_MAP,
    'mapeo_location': LOCATION_MAP,
    'location_derivada': True,
}

## Paso 4 (P5) — Nutriscore faltante

Ya convertido a nulo en el Paso 1 (era el literal `'Falta Dato'`, no un valor A–E
válido). Aquí solo se cuantifica el impacto: cuántos productos del catálogo quedan
sin calificación, y a cuántos *eventos* de `movements` afecta eso una vez unidos
(cada producto sin nutriscore aparece en muchos eventos).

In [ ]:
n_nutriscore_nulos_cat = int(catalog['nutriscore'].isna().sum())
print(f'Productos sin nutriscore: {n_nutriscore_nulos_cat} de {len(catalog)}')

log['transformaciones']['P5_nutriscore_faltante'] = {
    'valor_original': 'Falta Dato',
    'accion': 'convertido a null (nulo estructural preservado, no imputado)',
    'productos_afectados': n_nutriscore_nulos_cat,
}

## Paso 5 (P6) — Deduplicación de eventos

Clave de unicidad: `event_id` (UUID). Un duplicado significaría que el mismo
movimiento de inventario fue registrado dos veces — se conserva la primera
ocurrencia.

In [ ]:
n_antes_dedup = len(movements)
duplicados_detectados = int(movements['event_id'].duplicated().sum())
movements = movements.drop_duplicates(subset=['event_id'], keep='first')
n_duplicados = n_antes_dedup - len(movements)

print(f'Duplicados detectados (event_id): {duplicados_detectados}')
print(f'Movimientos después de deduplicación: {len(movements)} filas')

log['transformaciones']['P6_deduplicacion'] = {
    'duplicados_removidos': n_duplicados,
    'registros_despues': len(movements),
}

## Paso 6 (P3) — Integridad referencial

Todo evento en `movements` debe referenciar un `product_id` que exista en el
catálogo (tras el filtro P1). Un evento "huérfano" no se puede unir a datos
nutricionales y se descarta.

In [ ]:
n_mov_antes_p3 = len(movements)
mask_validos = movements['product_id'].isin(catalog['product_id'])
n_huerfanos = int((~mask_validos).sum())
movements = movements[mask_validos]
n_mov_post_p3 = len(movements)

print(f'Eventos huérfanos detectados: {n_huerfanos}')
print(f'Movimientos: {n_mov_antes_p3} -> {n_mov_post_p3}')

log['transformaciones']['P3_integridad_referencial'] = {
    'huerfanos_detectados': n_huerfanos,
    'huerfanos_removidos': n_huerfanos,
    'registros_movements_despues': n_mov_post_p3,
}

## Paso 7 — JOIN

`left join` de `movements` con las columnas nutricionales/derivadas del catálogo,
por `product_id`. Es un `left` (no `inner`) porque la integridad referencial ya se
garantizó en el paso anterior — no debería perderse ninguna fila aquí.

In [ ]:
inventory = movements.merge(
    catalog[['product_id', 'nutriscore', 'calories_100g', 'proteins_100g',
             'carbs_100g', 'category_name', 'location']],
    on='product_id', how='left',
)

print(f'Inventory (después del JOIN): {inventory.shape[0]} filas × {inventory.shape[1]} columnas')
assert len(inventory) == len(movements), 'El left join no debería cambiar el conteo de filas'

log['transformaciones']['join'] = {
    'tabla_izquierda': 'movements', 'tabla_derecha': 'catalog',
    'tipo_join': 'left', 'clave': 'product_id',
    'registros_resultado': len(inventory),
}

## Paso 8 — Derivadas y orden final de columnas

`action_type` es un alias normalizado de `event_type` (mismo valor, nombre
canónico usado río abajo). `dias_para_vencer = expiry_date - timestamp`: **no**
"hoy menos vencimiento" — tiene que ser relativo al momento del evento, para que
sea positivo cuando el producto seguía fresco en ese momento y negativo cuando ya
había vencido. Calcularlo contra la fecha actual del sistema haría que el
resultado cambiara cada vez que se ejecuta el notebook, rompiendo la
reproducibilidad y dando un número sin sentido de negocio.

In [ ]:
inventory['action_type'] = inventory['event_type'].astype(str)
inventory['dias_para_vencer'] = (
    inventory['expiry_date'].dt.normalize() - inventory['timestamp'].dt.normalize()
).dt.days

columnas_finales = [
    'event_id', 'household_id', 'stock_id', 'product_id', 'product_name',
    'action_type', 'quantity', 'timestamp', 'expiry_date', 'classification',
    'location', 'category_name', 'dias_para_vencer', 'nutriscore',
    'calories_100g', 'proteins_100g', 'carbs_100g',
]
inventory = inventory[columnas_finales]

print('Estructura final de inventory_v1:')
print(inventory.dtypes)
print(f'\nNulos por columna:')
print(inventory.isnull().sum())
print(f'\nEventos vencidos al momento del evento (dias_para_vencer < 0): {(inventory["dias_para_vencer"] < 0).sum()}')

log['transformaciones']['derivadas'] = {
    'action_type': 'alias de event_type (IN/OUT)',
    'dias_para_vencer': 'expiry_date - timestamp en dias (neg = vencido)',
    'columnas_finales': len(inventory.columns),
}

## Paso 9 — Exportación y bitácora

Se guarda a `inventory_v1_notebook_check.csv` (no a `inventory_v1.csv`) a propósito:
el archivo oficial ya lo produce `src/preprocessing.py` de forma reproducible y es
la fuente que consumen todos los demás notebooks y el dashboard. Este notebook
guarda su propio resultado con un sufijo distinto y lo **compara** contra el
oficial en la sección de verificación de abajo, en vez de competir por escribir el
mismo archivo.

In [ ]:
output_path = '../data/interim/inventory_v1_notebook_check.csv'
inventory.to_csv(output_path, index=False)
print(f'Archivo guardado: {output_path}')
print(f'  Tamaño: {len(inventory)} registros')

log['registros_finales'] = {
    'catalog': n_cat_post_p1,
    'movements': n_mov_post_p3,
    'inventory_v1': len(inventory),
}
log['calidad_datos'] = {
    'nulos_nutriscore_eventos': int(inventory['nutriscore'].isna().sum()),
    'eventos_vencidos_al_momento': int((inventory['dias_para_vencer'] < 0).sum()),
    'cardinalidad_category_name': int(inventory['category_name'].nunique()),
    'cardinalidad_location': int(inventory['location'].nunique()),
    'rango_temporal': {
        'min': str(inventory['timestamp'].min()),
        'max': str(inventory['timestamp'].max()),
    },
}

log_path = '../data/interim/transformations_log_notebook_check.json'
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=2, default=str)
print(f'Bitácora guardada: {log_path}')

print(f'\n=== RESUMEN DE TRANSFORMACIÓN ===')
print(f'Registros iniciales (movements): {log["registros_iniciales"]["movements"]}')
print(f'Registros finales (inventory_v1): {log["registros_finales"]["inventory_v1"]}')
print(f'Registros removidos: {log["registros_iniciales"]["movements"] - log["registros_finales"]["inventory_v1"]}')
print(f'Productos en catálogo final: {log["registros_finales"]["catalog"]}')

## Verificación — ¿coincide con el pipeline oficial?

Se compara este resultado (pandas, narrado paso a paso) contra
`data/interim/inventory_v1.csv`, el archivo oficial producido por
`src/preprocessing.py` (Polars). Si ambos coinciden fila por fila, es evidencia de
que la lógica de limpieza es correcta y está implementada consistentemente en dos
motores distintos — no es solo "el script corre", es "el script hace lo que dice
que hace".

In [ ]:
oficial = pd.read_csv('../data/interim/inventory_v1.csv')

checks = []
checks.append(('Filas', len(inventory), len(oficial), len(inventory) == len(oficial)))
checks.append(('Columnas', len(inventory.columns), len(oficial.columns), set(inventory.columns) == set(oficial.columns)))
checks.append(('Nutriscore nulos', int(inventory['nutriscore'].isna().sum()), int(oficial['nutriscore'].isna().sum()),
               inventory['nutriscore'].isna().sum() == oficial['nutriscore'].isna().sum()))
checks.append(('Eventos vencidos (dias_para_vencer<0)', int((inventory['dias_para_vencer'] < 0).sum()),
               int((oficial['dias_para_vencer'] < 0).sum()),
               (inventory['dias_para_vencer'] < 0).sum() == (oficial['dias_para_vencer'] < 0).sum()))

loc_nb = inventory['location'].value_counts().sort_index()
loc_of = oficial['location'].value_counts().sort_index()
checks.append(('Distribución location', loc_nb.to_dict(), loc_of.to_dict(), loc_nb.equals(loc_of)))

cat_nb = inventory['category_name'].value_counts().sort_index()
cat_of = oficial['category_name'].value_counts().sort_index()
checks.append(('Distribución category_name', cat_nb.to_dict(), cat_of.to_dict(), cat_nb.equals(cat_of)))

print(f"{'Check':<40}{'Notebook':<45}{'Oficial':<45}{'OK'}")
print('-' * 140)
todo_ok = True
for nombre, a, b, ok in checks:
    todo_ok = todo_ok and bool(ok)
    print(f'{nombre:<40}{str(a):<45}{str(b):<45}{"OK" if ok else "DIFIERE"}')

print()
if todo_ok:
    print('✓ El notebook reproduce exactamente los mismos resultados que src/preprocessing.py.')
else:
    print('⚠ Hay diferencias — revisar las filas marcadas DIFIERE arriba antes de confiar en este notebook como documentación del pipeline.')

assert todo_ok, 'La reimplementación en pandas no coincide con el pipeline oficial (src/preprocessing.py)'